In [10]:
# ==============================================================================
# PROJETO: Agregações e Estatísticas em PySpark
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, avg, max as spark_max, min as spark_min,
    variance, round as spark_round, year, month, desc, asc
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("AgregacoesVideosPySpark") \
    .getOrCreate()

In [11]:
# ------------------------------------------------------------------------------
# Etapa 1: Leia o arquivo 'videos-preparados.snappy.parquet' no dataframe 'df_video'
# ------------------------------------------------------------------------------
df_video = spark.read.parquet("videos-preparados.snappy.parquet")
df_video.printSchema()

# Garantias defensivas caso alguma coluna derivada não esteja no Parquet lido
if "Interaction" not in df_video.columns:
    df_video = df_video.withColumn("Interaction", col("Likes") + col("Comments") + col("Views"))

if "Year" not in df_video.columns:
    df_video = df_video.withColumn("Year", year(col("Published At")))

if "Month" not in df_video.columns:
    df_video = df_video.withColumn("Month", month(col("Published At")))

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)



In [12]:
# ------------------------------------------------------------------------------
# Etapa 2: Calcule a quantidade de registros para cada valor único da coluna "Keyword"
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(count("*").alias("Total_Registros")) \
    .show(truncate=False)

+----------------+---------------+
|Keyword         |Total_Registros|
+----------------+---------------+
|computer science|48             |
|lofi            |40             |
|finance         |39             |
|cnn             |50             |
|apple           |42             |
|news            |39             |
|mukbang         |45             |
|education       |24             |
|interview       |50             |
|crypto          |50             |
|mathchemistry   |15             |
|food            |48             |
|data science    |50             |
|trolling        |50             |
|tutorial        |50             |
|literature      |46             |
|sat             |49             |
|history         |49             |
|cubes           |49             |
|music           |46             |
+----------------+---------------+
only showing top 20 rows


In [13]:
# ------------------------------------------------------------------------------
# Etapa 3: Calcule a média da coluna "Interaction" para cada valor único da coluna 'Keyword'
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(avg("Interaction").alias("Media_Interaction")) \
    .show(truncate=False)

+----------------+--------------------+
|Keyword         |Media_Interaction   |
+----------------+--------------------+
|computer science|1226793.0208333333  |
|lofi            |4167085.875         |
|finance         |708542.9487179487   |
|cnn             |570650.86           |
|apple           |1.0873628214285715E7|
|news            |251688.71794871794  |
|mukbang         |1.1053630377777778E7|
|education       |2750838.625         |
|interview       |3044867.04          |
|crypto          |413676.2            |
|mathchemistry   |3427342.7333333334  |
|food            |5352944.104166667   |
|data science    |562465.28           |
|trolling        |1484584.88          |
|tutorial        |6936688.3           |
|literature      |881726.5            |
|sat             |1098927.0           |
|history         |1.565269257142857E7 |
|cubes           |1.5043961224489795E7|
|music           |2.9691370304347824E7|
+----------------+--------------------+
only showing top 20 rows


In [14]:
# ------------------------------------------------------------------------------
# Etapa 4: Calcule o valor máximo da coluna "Interaction" para cada valor único da coluna 'Keyword'
# e nomeie de 'Rank Interactions', em seguida ordene pela nova coluna em ordem decrescente
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(spark_max("Interaction").alias("Rank Interactions")) \
    .orderBy(desc("Rank Interactions")) \
    .show(truncate=False)

+--------+-----------------+
|Keyword |Rank Interactions|
+--------+-----------------+
|animals |1593623628       |
|music   |922551152        |
|bed     |532691631        |
|history |440187490        |
|apple   |429916936        |
|mrbeast |300397699        |
|google  |239385460        |
|business|210025196        |
|cubes   |170925917        |
|sports  |106924567        |
|mukbang |87433858         |
|lofi    |86445177         |
|tutorial|69616442         |
|movies  |65253870         |
|marvel  |56247330         |
|how-to  |53053975         |
|food    |48754479         |
|physics |43463298         |
|asmr    |34411125         |
|nintendo|32268486         |
+--------+-----------------+
only showing top 20 rows


In [15]:
# ------------------------------------------------------------------------------
# Etapa 5: Calcule a média e a variância da coluna 'Views' para cada valor único da coluna 'Keyword'
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(
        avg("Views").alias("Media_Views"),
        variance("Views").alias("Variancia_Views")
    ) \
    .show(truncate=False)

+----------------+--------------------+---------------------+
|Keyword         |Media_Views         |Variancia_Views      |
+----------------+--------------------+---------------------+
|computer science|1191958.7083333333  |2.81219868165102E12  |
|lofi            |4089363.0           |1.8462096414766778E14|
|finance         |694223.4358974359   |3.3044831750970425E12|
|cnn             |554240.38           |1.5634236184681186E11|
|apple           |1.0746930452380951E7|4.299927977442589E15 |
|news            |247492.1794871795   |1.0675125766725641E11|
|mukbang         |1.0904772355555555E7|5.5860732389731794E14|
|education       |2684432.8333333335  |1.8335722493392145E13|
|interview       |2966111.86          |1.819220996034335E13 |
|crypto          |404608.22           |3.513691634369074E12 |
|mathchemistry   |3328125.2666666666  |2.4914670652568496E13|
|food            |5252406.25          |7.326374128154842E13 |
|data science    |544771.98           |5.4793365253499945E11|
|trollin

In [16]:
# ------------------------------------------------------------------------------
# Etapa 6: Calcule a média, o valor mínimo e o valor máximo de 'Views' para cada valor
# único da coluna 'Keyword', sem casas decimais
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(
        spark_round(avg("Views"), 0).cast("long").alias("Media_Views"),
        spark_round(spark_min("Views"), 0).cast("long").alias("Min_Views"),
        spark_round(spark_max("Views"), 0).cast("long").alias("Max_Views")
    ) \
    .show(truncate=False)

+----------------+-----------+---------+---------+
|Keyword         |Media_Views|Min_Views|Max_Views|
+----------------+-----------+---------+---------+
|computer science|1191959    |16115    |7004107  |
|lofi            |4089363    |6817     |84747957 |
|finance         |694223     |1195     |9450554  |
|cnn             |554240     |51269    |1889320  |
|apple           |10746930   |1954     |425478119|
|news            |247492     |10642    |1465011  |
|mukbang         |10904772   |3618     |86169225 |
|education       |2684433    |6611     |17103736 |
|interview       |2966112    |2587     |22529756 |
|crypto          |404608     |1599     |11805668 |
|mathchemistry   |3328125    |25       |18496859 |
|food            |5252406    |47430    |48018833 |
|data science    |544772     |911      |3069097  |
|trolling        |1420141    |5388     |14286302 |
|tutorial        |6761032    |19323    |68512549 |
|literature      |863021     |2847     |4231789  |
|sat             |1065869    |7

In [17]:
# ------------------------------------------------------------------------------
# Etapa 7: Mostre o primeiro e o último 'Published At' para cada valor único da coluna 'Keyword'
# ------------------------------------------------------------------------------
df_video.groupBy("Keyword") \
    .agg(
        spark_min("Published At").alias("Primeiro_Published_At"),
        spark_max("Published At").alias("Ultimo_Published_At")
    ) \
    .show(truncate=False)

+----------------+---------------------+-------------------+
|Keyword         |Primeiro_Published_At|Ultimo_Published_At|
+----------------+---------------------+-------------------+
|computer science|2009-08-20           |2022-08-12         |
|lofi            |2019-12-08           |2022-08-24         |
|finance         |2012-11-27           |2022-08-24         |
|cnn             |2022-07-14           |2022-08-24         |
|apple           |2016-11-02           |2022-08-24         |
|news            |2022-08-18           |2022-08-24         |
|mukbang         |2020-02-29           |2022-08-24         |
|education       |2008-07-25           |2022-08-24         |
|interview       |2016-01-05           |2022-08-24         |
|crypto          |2022-03-11           |2022-08-24         |
|mathchemistry   |2013-04-15           |2022-05-03         |
|food            |2017-05-31           |2022-08-24         |
|data science    |2018-06-23           |2022-08-24         |
|trolling        |2020-0

In [18]:
# ------------------------------------------------------------------------------
# Etapa 8: Conte todos os 'title' de forma normal e todos os únicos e verifique se há diferença
# ------------------------------------------------------------------------------
coluna_titulo = "Title" if "Title" in df_video.columns else "title"

contagem_titulos = df_video.select(
    count(col(coluna_titulo)).alias("Total_Titulos"),
    countDistinct(col(coluna_titulo)).alias("Titulos_Unicos")
).collect()[0]

total = contagem_titulos["Total_Titulos"]
unicos = contagem_titulos["Titulos_Unicos"]
diferenca = total - unicos

print(f"Total de títulos: {total}")
print(f"Títulos únicos:   {unicos}")
print(f"Diferença:        {diferenca} ({'Há títulos repetidos' if diferenca > 0 else 'Não há diferença'})")

Total de títulos: 1869
Títulos únicos:   1854
Diferença:        15 (Há títulos repetidos)


In [19]:
# ------------------------------------------------------------------------------
# Etapa 9: Mostre a quantidade de registros ordenados por ano em ordem ascendente
# ------------------------------------------------------------------------------
df_video.groupBy("Year") \
    .agg(count("*").alias("Quantidade_Registros")) \
    .orderBy(asc("Year")) \
    .show(truncate=False)

+----+--------------------+
|Year|Quantidade_Registros|
+----+--------------------+
|2007|2                   |
|2008|1                   |
|2009|9                   |
|2010|6                   |
|2011|4                   |
|2012|12                  |
|2013|6                   |
|2014|10                  |
|2015|15                  |
|2016|34                  |
|2017|47                  |
|2018|57                  |
|2019|86                  |
|2020|158                 |
|2021|229                 |
|2022|1193                |
+----+--------------------+



In [20]:
# ------------------------------------------------------------------------------
# Etapa 10: Mostre a quantidade de registros ordenados por ano e mês em ordem ascendente
# ------------------------------------------------------------------------------
df_video.groupBy("Year", "Month") \
    .agg(count("*").alias("Quantidade_Registros")) \
    .orderBy(asc("Year"), asc("Month")) \
    .show(50, truncate=False)

+----+-----+--------------------+
|Year|Month|Quantidade_Registros|
+----+-----+--------------------+
|2007|7    |1                   |
|2007|12   |1                   |
|2008|7    |1                   |
|2009|2    |2                   |
|2009|6    |2                   |
|2009|7    |1                   |
|2009|8    |1                   |
|2009|10   |1                   |
|2009|12   |2                   |
|2010|3    |1                   |
|2010|5    |2                   |
|2010|6    |1                   |
|2010|9    |1                   |
|2010|10   |1                   |
|2011|2    |1                   |
|2011|5    |1                   |
|2011|9    |1                   |
|2011|10   |1                   |
|2012|1    |1                   |
|2012|2    |3                   |
|2012|3    |2                   |
|2012|4    |1                   |
|2012|6    |1                   |
|2012|8    |1                   |
|2012|10   |1                   |
|2012|11   |2                   |
|2013|3    |1 

In [21]:
# ------------------------------------------------------------------------------
# Etapa 11: Calcule a média acumulativa de 'Likes' para cada 'Keyword' ao longo dos anos
# ------------------------------------------------------------------------------
df_likes_ano = df_video.groupBy("Keyword", "Year") \
    .agg(avg("Likes").alias("Media_Anual_Likes"))

window_acumulada = Window.partitionBy("Keyword") \
    .orderBy(asc("Year")) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_likes_acumulado = df_likes_ano.withColumn(
    "Media_Acumulativa_Likes",
    spark_round(avg("Media_Anual_Likes").over(window_acumulada), 2)
)

df_likes_acumulado.orderBy("Keyword", "Year").show(50, truncate=False)

+----------------+----+------------------+-----------------------+
|Keyword         |Year|Media_Anual_Likes |Media_Acumulativa_Likes|
+----------------+----+------------------+-----------------------+
|animals         |2009|1357197.0         |1357197.0              |
|animals         |2010|203367.0          |780282.0               |
|animals         |2013|1.1025176E7       |4195246.67             |
|animals         |2014|3381630.0         |3991842.5              |
|animals         |2019|1103713.0         |3414216.6              |
|animals         |2020|769652.1111111111 |2973455.85             |
|animals         |2021|112729.75         |2564780.69             |
|animals         |2022|30335.214285714286|2247975.01             |
|apple           |2016|4144389.0         |4144389.0              |
|apple           |2021|38261.0           |2091325.0              |
|apple           |2022|19416.6           |1400688.87             |
|asmr            |2020|148120.0          |148120.0            